In [ ]:
!pip install torch transformers datasets -q
print("Packages installed")

Packages installed


In [ ]:
from datasets import load_dataset
print("Downloading WikiText-103 (about 2 minutes)...")
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")
print("Done. Dataset cached.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Done. Dataset cached.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import json

# -------------------- CONFIG --------------------
class Config:
    hidden_size = 256
    num_layers = 2
    num_heads = 4
    num_experts = 8
    expert_hidden = 512
    max_seq_len = 128
    vocab_size = 50257

    d_embed = 8
    ema_alpha = 0.99
    sinkhorn_iters = 5
    sinkhorn_temp = 0.1
    capacity_tau = 0.1
    capacity_slack = 0.1

    batch_size = 4
    warmup_steps = 2000
    bbr_steps = 8000
    total_steps = 10000
    lr = 3e-4
    grad_clip = 1.0

    device = "cuda"
    log_every = 500
    seed = 42

cfg = Config()

# -------------------- SINKHORN --------------------
def sinkhorn(S, n_iters, capacity, temp):
    T, E = S.shape
    P = torch.exp(S / temp)
    for _ in range(n_iters):
        P = P / (P.sum(dim=1, keepdim=True) + 1e-8)
        col_sum = P.sum(dim=0)
        scale = torch.minimum(capacity / (col_sum + 1e-8), torch.ones_like(col_sum))
        P = P * scale.unsqueeze(0)
    return P

# -------------------- ROUTERS --------------------
class StandardRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.gate = nn.Linear(cfg.hidden_size, cfg.num_experts, bias=False)

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        indices = probs.argmax(dim=-1)
        T = x.shape[0]
        E = self.cfg.num_experts
        f = torch.zeros(E, device=x.device)
        for e in range(E):
            f[e] = (indices == e).float().mean()
        p = probs.mean(dim=0)
        aux_loss = E * (f * p).sum()
        return indices, probs, aux_loss

class BBRRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        E = cfg.num_experts
        d = cfg.d_embed
        self.token_proj = nn.Linear(cfg.hidden_size, d, bias=False)
        self.register_buffer('prototypes', F.normalize(torch.randn(E, d), dim=-1))
        self.register_buffer('confidence_ema', torch.ones(E) * 0.5)

    def get_capacity(self, T):
        E = self.cfg.num_experts
        base = T / E * (1 + self.cfg.capacity_slack)
        inv_conf = 2.0 - F.softmax(self.confidence_ema / self.cfg.capacity_tau, dim=0) * E
        capacity = base * inv_conf
        return capacity.clamp(base * 0.5, base * 2.0)

    def forward(self, x, hard=False):
        T, H = x.shape
        E = self.cfg.num_experts
        tok_emb = F.normalize(self.token_proj(x), dim=-1)
        proto_emb = F.normalize(self.prototypes, dim=-1)
        S = tok_emb @ proto_emb.T
        capacity = self.get_capacity(T)
        P = sinkhorn(S, self.cfg.sinkhorn_iters, capacity, self.cfg.sinkhorn_temp)

        with torch.no_grad():
            indices_hard = P.argmax(dim=-1)
            alpha = self.cfg.ema_alpha
            for e in range(E):
                mask = (indices_hard == e)
                if mask.any():
                    new_proto = F.normalize(x[mask].mean(dim=0), dim=-1)
                    self.prototypes[e] = alpha * self.prototypes[e] + (1 - alpha) * new_proto
                    self.prototypes[e] = F.normalize(self.prototypes[e], dim=-1)
                conf_e = P[:, e].max().item() if P[:, e].max() > 0 else 0.0
                self.confidence_ema[e] = alpha * self.confidence_ema[e] + (1 - alpha) * conf_e

        proto_n = F.normalize(self.prototypes, dim=-1)
        sim = proto_n @ proto_n.T
        mask_off = 1 - torch.eye(E, device=x.device)
        collapse_loss = (sim * mask_off).sum() / (E * (E - 1))

        if hard:
            return P.argmax(dim=-1), collapse_loss
        return P, collapse_loss

# -------------------- MoE LAYER --------------------
class Expert(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.hidden_size, cfg.expert_hidden),
            nn.GELU(),
            nn.Linear(cfg.expert_hidden, cfg.hidden_size)
        )
    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.experts = nn.ModuleList([Expert(cfg) for _ in range(cfg.num_experts)])
        if router_type == "standard":
            self.router = StandardRouter(cfg)
        else:
            self.router = BBRRouter(cfg)
        self.router_type = router_type

    def forward(self, x, step=0):
        B, L, H = x.shape
        x_flat = x.view(B*L, H)
        if self.router_type == "standard":
            indices, probs, aux_loss = self.router(x_flat)
            out = torch.zeros_like(x_flat)
            for e in range(self.cfg.num_experts):
                mask = (indices == e)
                if mask.any():
                    out[mask] = self.experts[e](x_flat[mask])
            return out.view(B, L, H), aux_loss, indices
        else:  # bbr
            use_soft = (step >= self.cfg.warmup_steps)
            if use_soft:
                P, aux_loss = self.router(x_flat, hard=False)
                # top-2 to save memory
                top_weights, top_indices = P.topk(2, dim=-1)
                out = torch.zeros_like(x_flat)
                for k in range(2):
                    e_idx = top_indices[:, k]
                    w = top_weights[:, k:k+1]
                    for e in range(self.cfg.num_experts):
                        mask = (e_idx == e)
                        if mask.any():
                            out[mask] += w[mask] * self.experts[e](x_flat[mask])
                indices = P.argmax(dim=-1)
            else:
                indices, aux_loss = self.router(x_flat, hard=True)
                out = torch.zeros_like(x_flat)
                for e in range(self.cfg.num_experts):
                    mask = (indices == e)
                    if mask.any():
                        out[mask] = self.experts[e](x_flat[mask])
            return out.view(B, L, H), aux_loss, indices

# -------------------- TRANSFORMER BLOCK WITH CAUSAL MASK --------------------
class TransformerBlock(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, batch_first=True)
        self.moe = MoELayer(cfg, router_type)
        self.ln1 = nn.LayerNorm(cfg.hidden_size)
        self.ln2 = nn.LayerNorm(cfg.hidden_size)

    def forward(self, x, step):
        L = x.shape[1]
        causal_mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = self.ln1(x + attn_out)
        moe_out, aux_loss, indices = self.moe(x, step)
        x = self.ln2(x + moe_out)
        return x, aux_loss, indices

class MoETransformer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_embed = nn.Embedding(cfg.max_seq_len, cfg.hidden_size)
        self.blocks = nn.ModuleList([TransformerBlock(cfg, router_type) for _ in range(cfg.num_layers)])
        self.ln_f = nn.LayerNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

    def forward(self, input_ids, step=0):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos_embed(pos)
        total_aux = 0.0
        all_indices = []
        for block in self.blocks:
            x, aux, indices = block(x, step)
            total_aux += aux
            all_indices.append(indices)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, total_aux, all_indices

# -------------------- METRICS --------------------
def expert_utilization_entropy(all_indices, num_experts):
    counts = torch.zeros(num_experts)
    for indices in all_indices:
        for e in range(num_experts):
            counts[e] += (indices == e).float().sum()
    probs = counts / counts.sum()
    probs = probs.clamp(min=1e-8)
    entropy = -(probs * probs.log()).sum()
    max_entropy = torch.tensor(num_experts).float().log()
    return (entropy / max_entropy).item()

# -------------------- DATA LOADER --------------------
def get_dataloader(cfg, tokenizer, split="train"):
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split=split)
    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=cfg.max_seq_len,
                         padding="max_length", return_tensors="pt")
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
    tokenized.set_format("torch")
    return DataLoader(tokenized, batch_size=cfg.batch_size, shuffle=True, drop_last=True)

# -------------------- TRAINING --------------------
def train(router_type, cfg):
    torch.manual_seed(cfg.seed)
    print(f"\n{'='*50}")
    print(f"Training {router_type.upper()} router")
    print(f"{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = MoETransformer(cfg, router_type).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.total_steps)

    train_loader = get_dataloader(cfg, tokenizer, "train")
    val_loader   = get_dataloader(cfg, tokenizer, "validation")

    logs = []
    step = 0
    data_iter = iter(train_loader)

    while step < cfg.total_steps:
        model.train()
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(cfg.device)
        targets = input_ids[:, 1:].contiguous()
        inputs  = input_ids[:, :-1].contiguous()

        logits, aux_loss, all_indices = model(inputs, step=step)
        lm_loss = F.cross_entropy(logits.view(-1, cfg.vocab_size),
                                  targets.view(-1),
                                  ignore_index=tokenizer.pad_token_id)
        loss = lm_loss + 0.01 * aux_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()
        step += 1

        if step % cfg.log_every == 0 or step == cfg.total_steps:
            model.eval()
            val_losses = []
            val_entropies = []
            with torch.no_grad():
                for i, vbatch in enumerate(val_loader):
                    if i >= 10: break
                    vids = vbatch["input_ids"].to(cfg.device)
                    vi, vt = vids[:, :-1], vids[:, 1:]
                    vlogits, _, vindices = model(vi, step=step)
                    vloss = F.cross_entropy(vlogits.view(-1, cfg.vocab_size),
                                            vt.reshape(-1),
                                            ignore_index=tokenizer.pad_token_id)
                    val_losses.append(vloss.item())
                    val_entropies.append(expert_utilization_entropy(vindices, cfg.num_experts))
            ppl = np.exp(np.mean(val_losses))
            ent = np.mean(val_entropies)
            phase = "warmup" if step < cfg.warmup_steps else "bbr"
            logs.append({"step": step, "router": router_type, "perplexity": round(ppl,2),
                         "entropy": round(ent,4), "phase": phase})
            print(f"Step {step:5d} | PPL: {ppl:.2f} | Entropy: {ent:.4f} | {phase}")

    return logs

# -------------------- RUN --------------------
print("Starting Standard Router (this will take ~20 minutes)...")
logs_std = train("standard", cfg)
print("\nStarting BBR Router (another ~20 minutes)...")
logs_bbr = train("bbr", cfg)

results = {"standard": logs_std, "bbr": logs_bbr}
with open("/content/bbr_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*50)
print("Experiment Complete. Final comparison:")
print("Router    | Final Perplexity | Final Entropy")
print("-"*45)
for r in ["standard", "bbr"]:
    last = results[r][-1]
    print(f"{r:<9} | {last['perplexity']:>16.2f} | {last['entropy']:>12.4f}")

Starting Standard Router (this will take ~20 minutes)...

Training STANDARD router


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import json

# -------------------- CONFIG --------------------
class Config:
    hidden_size = 256
    num_layers = 2
    num_heads = 4
    num_experts = 8
    expert_hidden = 512
    max_seq_len = 128
    vocab_size = 50257

    d_embed = 8
    ema_alpha = 0.99
    sinkhorn_iters = 5
    sinkhorn_temp = 0.1
    capacity_tau = 0.1
    capacity_slack = 0.1

    batch_size = 4
    warmup_steps = 2000
    bbr_steps = 8000
    total_steps = 10000
    lr = 3e-4
    grad_clip = 1.0

    device = "cuda"
    log_every = 500
    seed = 42

cfg = Config()

# -------------------- SINKHORN --------------------
def sinkhorn(S, n_iters, capacity, temp):
    T, E = S.shape
    P = torch.exp(S / temp)
    for _ in range(n_iters):
        P = P / (P.sum(dim=1, keepdim=True) + 1e-8)
        col_sum = P.sum(dim=0)
        scale = torch.minimum(capacity / (col_sum + 1e-8), torch.ones_like(col_sum))
        P = P * scale.unsqueeze(0)
    return P

# -------------------- ROUTERS --------------------
class StandardRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.gate = nn.Linear(cfg.hidden_size, cfg.num_experts, bias=False)

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        indices = probs.argmax(dim=-1)
        T = x.shape[0]
        E = self.cfg.num_experts
        f = torch.zeros(E, device=x.device)
        for e in range(E):
            f[e] = (indices == e).float().mean()
        p = probs.mean(dim=0)
        aux_loss = E * (f * p).sum()
        return indices, probs, aux_loss

class BBRRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        E = cfg.num_experts
        d = cfg.d_embed
        self.token_proj = nn.Linear(cfg.hidden_size, d, bias=False)
        self.register_buffer('prototypes', F.normalize(torch.randn(E, d), dim=-1))
        self.register_buffer('confidence_ema', torch.ones(E) * 0.5)

    def get_capacity(self, T):
        E = self.cfg.num_experts
        base = T / E * (1 + self.cfg.capacity_slack)
        inv_conf = 2.0 - F.softmax(self.confidence_ema / self.cfg.capacity_tau, dim=0) * E
        capacity = base * inv_conf
        return capacity.clamp(base * 0.5, base * 2.0)

    def forward(self, x, hard=False):
        T, H = x.shape
        E = self.cfg.num_experts
        tok_emb = F.normalize(self.token_proj(x), dim=-1)
        proto_emb = F.normalize(self.prototypes, dim=-1)
        S = tok_emb @ proto_emb.T
        capacity = self.get_capacity(T)
        P = sinkhorn(S, self.cfg.sinkhorn_iters, capacity, self.cfg.sinkhorn_temp)

        with torch.no_grad():
            indices_hard = P.argmax(dim=-1)
            alpha = self.cfg.ema_alpha
            for e in range(E):
                mask = (indices_hard == e)
                if mask.any():
                    new_proto = F.normalize(x[mask].mean(dim=0), dim=-1)
                    self.prototypes[e] = alpha * self.prototypes[e] + (1 - alpha) * new_proto
                    self.prototypes[e] = F.normalize(self.prototypes[e], dim=-1)
                conf_e = P[:, e].max().item() if P[:, e].max() > 0 else 0.0
                self.confidence_ema[e] = alpha * self.confidence_ema[e] + (1 - alpha) * conf_e

        proto_n = F.normalize(self.prototypes, dim=-1)
        sim = proto_n @ proto_n.T
        mask_off = 1 - torch.eye(E, device=x.device)
        collapse_loss = (sim * mask_off).sum() / (E * (E - 1))

        if hard:
            return P.argmax(dim=-1), collapse_loss
        return P, collapse_loss

# -------------------- MoE LAYER --------------------
class Expert(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.hidden_size, cfg.expert_hidden),
            nn.GELU(),
            nn.Linear(cfg.expert_hidden, cfg.hidden_size)
        )
    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.experts = nn.ModuleList([Expert(cfg) for _ in range(cfg.num_experts)])
        if router_type == "standard":
            self.router = StandardRouter(cfg)
        else:
            self.router = BBRRouter(cfg)
        self.router_type = router_type

    def forward(self, x, step=0):
        B, L, H = x.shape
        x_flat = x.view(B*L, H)
        if self.router_type == "standard":
            indices, probs, aux_loss = self.router(x_flat)
            out = torch.zeros_like(x_flat)
            for e in range(self.cfg.num_experts):
                mask = (indices == e)
                if mask.any():
                    out[mask] = self.experts[e](x_flat[mask])
            return out.view(B, L, H), aux_loss, indices
        else:  # bbr
            use_soft = (step >= self.cfg.warmup_steps)
            if use_soft:
                P, aux_loss = self.router(x_flat, hard=False)
                # top-2 to save memory
                top_weights, top_indices = P.topk(2, dim=-1)
                out = torch.zeros_like(x_flat)
                for k in range(2):
                    e_idx = top_indices[:, k]
                    w = top_weights[:, k:k+1]
                    for e in range(self.cfg.num_experts):
                        mask = (e_idx == e)
                        if mask.any():
                            out[mask] += w[mask] * self.experts[e](x_flat[mask])
                indices = P.argmax(dim=-1)
            else:
                indices, aux_loss = self.router(x_flat, hard=True)
                out = torch.zeros_like(x_flat)
                for e in range(self.cfg.num_experts):
                    mask = (indices == e)
                    if mask.any():
                        out[mask] = self.experts[e](x_flat[mask])
            return out.view(B, L, H), aux_loss, indices

# -------------------- TRANSFORMER BLOCK WITH CAUSAL MASK --------------------
class TransformerBlock(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, batch_first=True)
        self.moe = MoELayer(cfg, router_type)
        self.ln1 = nn.LayerNorm(cfg.hidden_size)
        self.ln2 = nn.LayerNorm(cfg.hidden_size)

    def forward(self, x, step):
        L = x.shape[1]
        causal_mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = self.ln1(x + attn_out)
        moe_out, aux_loss, indices = self.moe(x, step)
        x = self.ln2(x + moe_out)
        return x, aux_loss, indices

class MoETransformer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_embed = nn.Embedding(cfg.max_seq_len, cfg.hidden_size)
        self.blocks = nn.ModuleList([TransformerBlock(cfg, router_type) for _ in range(cfg.num_layers)])
        self.ln_f = nn.LayerNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

    def forward(self, input_ids, step=0):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos_embed(pos)
        total_aux = 0.0
        all_indices = []
        for block in self.blocks:
            x, aux, indices = block(x, step)
            total_aux += aux
            all_indices.append(indices)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, total_aux, all_indices

# -------------------- METRICS (FIXED DEVICE) --------------------
def expert_utilization_entropy(all_indices, num_experts):
    if not all_indices:
        return 0.0
    device = all_indices[0].device
    counts = torch.zeros(num_experts, device=device)
    for indices in all_indices:
        for e in range(num_experts):
            counts[e] += (indices == e).float().sum()
    probs = counts / counts.sum()
    probs = probs.clamp(min=1e-8)
    entropy = -(probs * probs.log()).sum()
    max_entropy = torch.tensor(num_experts, device=device).float().log()
    return (entropy / max_entropy).item()

# -------------------- DATA LOADER --------------------
def get_dataloader(cfg, tokenizer, split="train"):
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split=split)
    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=cfg.max_seq_len,
                         padding="max_length", return_tensors="pt")
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
    tokenized.set_format("torch")
    return DataLoader(tokenized, batch_size=cfg.batch_size, shuffle=True, drop_last=True)

# -------------------- TRAINING --------------------
def train(router_type, cfg):
    torch.manual_seed(cfg.seed)
    print(f"\n{'='*50}")
    print(f"Training {router_type.upper()} router")
    print(f"{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = MoETransformer(cfg, router_type).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.total_steps)

    train_loader = get_dataloader(cfg, tokenizer, "train")
    val_loader   = get_dataloader(cfg, tokenizer, "validation")

    logs = []
    step = 0
    data_iter = iter(train_loader)

    while step < cfg.total_steps:
        model.train()
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(cfg.device)
        targets = input_ids[:, 1:].contiguous()
        inputs  = input_ids[:, :-1].contiguous()

        logits, aux_loss, all_indices = model(inputs, step=step)
        lm_loss = F.cross_entropy(logits.view(-1, cfg.vocab_size),
                                  targets.view(-1),
                                  ignore_index=tokenizer.pad_token_id)
        loss = lm_loss + 0.01 * aux_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()
        step += 1

        if step % cfg.log_every == 0 or step == cfg.total_steps:
            model.eval()
            val_losses = []
            val_entropies = []
            with torch.no_grad():
                for i, vbatch in enumerate(val_loader):
                    if i >= 10: break
                    vids = vbatch["input_ids"].to(cfg.device)
                    vi, vt = vids[:, :-1], vids[:, 1:]
                    vlogits, _, vindices = model(vi, step=step)
                    vloss = F.cross_entropy(vlogits.view(-1, cfg.vocab_size),
                                            vt.reshape(-1),
                                            ignore_index=tokenizer.pad_token_id)
                    val_losses.append(vloss.item())
                    val_entropies.append(expert_utilization_entropy(vindices, cfg.num_experts))
            ppl = np.exp(np.mean(val_losses))
            ent = np.mean(val_entropies)
            phase = "warmup" if step < cfg.warmup_steps else "bbr"
            logs.append({"step": step, "router": router_type, "perplexity": round(ppl,2),
                         "entropy": round(ent,4), "phase": phase})
            print(f"Step {step:5d} | PPL: {ppl:.2f} | Entropy: {ent:.4f} | {phase}")

    return logs

# -------------------- RUN --------------------
print("Starting Standard Router (this will take ~20 minutes)...")
logs_std = train("standard", cfg)
print("\nStarting BBR Router (another ~20 minutes)...")
logs_bbr = train("bbr", cfg)

results = {"standard": logs_std, "bbr": logs_bbr}
with open("/content/bbr_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*50)
print("Experiment Complete. Final comparison:")
print("Router    | Final Perplexity | Final Entropy")
print("-"*45)
for r in ["standard", "bbr"]:
    last = results[r][-1]
    print(f"{r:<9} | {last['perplexity']:>16.2f} | {last['entropy']:>12.4f}")

Starting Standard Router (this will take ~20 minutes)...

Training STANDARD router


Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Step   500 | PPL: 1352.57 | Entropy: 0.9957 | warmup
Step  1000 | PPL: 1395.59 | Entropy: 0.9971 | warmup
Step  1500 | PPL: 824.42 | Entropy: 0.9963 | warmup
Step  2000 | PPL: 371.66 | Entropy: 0.9981 | bbr
Step  2500 | PPL: 590.12 | Entropy: 0.9981 | bbr
Step  3000 | PPL: 492.80 | Entropy: 0.9980 | bbr
Step  3500 | PPL: 492.13 | Entropy: 0.9970 | bbr
Step  4000 | PPL: 405.49 | Entropy: 0.9960 | bbr
Step  4500 | PPL: nan | Entropy: 0.9974 | bbr
Step  5000 | PPL: 512.61 | Entropy: 0.9987 | bbr
Step  5500 | PPL: 258.49 | Entropy: 0.9971 | bbr
Step  6000 | PPL: 366.99 | Entropy: 0.9989 | bbr
Step  6500 | PPL: 842.73 | Entropy: 0.9988 | bbr
Step  7000 | PPL: 684.05 | Entropy: 0.9990 | bbr
Step  7500 | PPL: 312.17 | Entropy: 0.9990 | bbr
Step  8000 | PPL: 431.75 | Entropy: 0.9990 | bbr
Step  8500 | PPL: 456.10 | Entropy: 0.9993 | bbr
Step  9000 | PPL: 370.05 | Entropy: 0.9990 | bbr
Step  9500 | PPL: 431.34 | Entropy: 0.9993 | bbr
Step 10000 | PPL: 247.55 | Entropy: 0.9990 | bbr

Starting BB

RuntimeError: The size of tensor a (8) must match the size of tensor b (256) at non-singleton dimension 0

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import json

# -------------------- CONFIG --------------------
class Config:
    hidden_size = 256
    num_layers = 2
    num_heads = 4
    num_experts = 8
    expert_hidden = 512
    max_seq_len = 128
    vocab_size = 50257

    d_embed = 8
    ema_alpha = 0.99
    sinkhorn_iters = 5
    sinkhorn_temp = 0.1
    capacity_tau = 0.1
    capacity_slack = 0.1

    batch_size = 4
    warmup_steps = 2000
    bbr_steps = 8000
    total_steps = 10000
    lr = 3e-4
    grad_clip = 1.0

    device = "cuda"
    log_every = 500
    seed = 42

cfg = Config()

# -------------------- SINKHORN --------------------
def sinkhorn(S, n_iters, capacity, temp):
    T, E = S.shape
    P = torch.exp(S / temp)
    for _ in range(n_iters):
        P = P / (P.sum(dim=1, keepdim=True) + 1e-8)
        col_sum = P.sum(dim=0)
        scale = torch.minimum(capacity / (col_sum + 1e-8), torch.ones_like(col_sum))
        P = P * scale.unsqueeze(0)
    return P

# -------------------- ROUTERS --------------------
class StandardRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.gate = nn.Linear(cfg.hidden_size, cfg.num_experts, bias=False)

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        indices = probs.argmax(dim=-1)
        T = x.shape[0]
        E = self.cfg.num_experts
        f = torch.zeros(E, device=x.device)
        for e in range(E):
            f[e] = (indices == e).float().mean()
        p = probs.mean(dim=0)
        aux_loss = E * (f * p).sum()
        return indices, probs, aux_loss

class BBRRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        E = cfg.num_experts
        d = cfg.d_embed
        self.token_proj = nn.Linear(cfg.hidden_size, d, bias=False)
        self.register_buffer('prototypes', F.normalize(torch.randn(E, d), dim=-1))
        self.register_buffer('confidence_ema', torch.ones(E) * 0.5)

    def get_capacity(self, T):
        E = self.cfg.num_experts
        base = T / E * (1 + self.cfg.capacity_slack)
        inv_conf = 2.0 - F.softmax(self.confidence_ema / self.cfg.capacity_tau, dim=0) * E
        capacity = base * inv_conf
        return capacity.clamp(base * 0.5, base * 2.0)

    def forward(self, x, hard=False):
        T, H = x.shape
        E = self.cfg.num_experts
        # Project tokens to embedding space
        tok_emb = F.normalize(self.token_proj(x), dim=-1)   # [T, d]
        proto_emb = F.normalize(self.prototypes, dim=-1)    # [E, d]
        S = tok_emb @ proto_emb.T                           # [T, E]
        capacity = self.get_capacity(T)
        P = sinkhorn(S, self.cfg.sinkhorn_iters, capacity, self.cfg.sinkhorn_temp)

        with torch.no_grad():
            indices_hard = P.argmax(dim=-1)                 # [T]
            alpha = self.cfg.ema_alpha
            for e in range(E):
                mask = (indices_hard == e)
                if mask.any():
                    # Use projected token embeddings, not raw x
                    new_proto = tok_emb[mask].mean(dim=0)   # [d]
                    new_proto = F.normalize(new_proto, dim=-1)
                    self.prototypes[e] = alpha * self.prototypes[e] + (1 - alpha) * new_proto
                    self.prototypes[e] = F.normalize(self.prototypes[e], dim=-1)
                # Update confidence EMA using max probability for this expert
                conf_e = P[:, e].max().item() if P[:, e].max() > 0 else 0.0
                self.confidence_ema[e] = alpha * self.confidence_ema[e] + (1 - alpha) * conf_e

        # Anti-collapse loss: push prototypes apart
        proto_n = F.normalize(self.prototypes, dim=-1)
        sim = proto_n @ proto_n.T
        mask_off = 1 - torch.eye(E, device=x.device)
        collapse_loss = (sim * mask_off).sum() / (E * (E - 1))

        if hard:
            return P.argmax(dim=-1), collapse_loss
        return P, collapse_loss

# -------------------- MoE LAYER --------------------
class Expert(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.hidden_size, cfg.expert_hidden),
            nn.GELU(),
            nn.Linear(cfg.expert_hidden, cfg.hidden_size)
        )
    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.experts = nn.ModuleList([Expert(cfg) for _ in range(cfg.num_experts)])
        if router_type == "standard":
            self.router = StandardRouter(cfg)
        else:
            self.router = BBRRouter(cfg)
        self.router_type = router_type

    def forward(self, x, step=0):
        B, L, H = x.shape
        x_flat = x.view(B*L, H)
        if self.router_type == "standard":
            indices, probs, aux_loss = self.router(x_flat)
            out = torch.zeros_like(x_flat)
            for e in range(self.cfg.num_experts):
                mask = (indices == e)
                if mask.any():
                    out[mask] = self.experts[e](x_flat[mask])
            return out.view(B, L, H), aux_loss, indices
        else:  # bbr
            use_soft = (step >= self.cfg.warmup_steps)
            if use_soft:
                P, aux_loss = self.router(x_flat, hard=False)
                # top-2 to save memory
                top_weights, top_indices = P.topk(2, dim=-1)
                out = torch.zeros_like(x_flat)
                for k in range(2):
                    e_idx = top_indices[:, k]
                    w = top_weights[:, k:k+1]
                    for e in range(self.cfg.num_experts):
                        mask = (e_idx == e)
                        if mask.any():
                            out[mask] += w[mask] * self.experts[e](x_flat[mask])
                indices = P.argmax(dim=-1)
            else:
                indices, aux_loss = self.router(x_flat, hard=True)
                out = torch.zeros_like(x_flat)
                for e in range(self.cfg.num_experts):
                    mask = (indices == e)
                    if mask.any():
                        out[mask] = self.experts[e](x_flat[mask])
            return out.view(B, L, H), aux_loss, indices

# -------------------- TRANSFORMER BLOCK WITH CAUSAL MASK --------------------
class TransformerBlock(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, batch_first=True)
        self.moe = MoELayer(cfg, router_type)
        self.ln1 = nn.LayerNorm(cfg.hidden_size)
        self.ln2 = nn.LayerNorm(cfg.hidden_size)

    def forward(self, x, step):
        L = x.shape[1]
        causal_mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = self.ln1(x + attn_out)
        moe_out, aux_loss, indices = self.moe(x, step)
        x = self.ln2(x + moe_out)
        return x, aux_loss, indices

class MoETransformer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_embed = nn.Embedding(cfg.max_seq_len, cfg.hidden_size)
        self.blocks = nn.ModuleList([TransformerBlock(cfg, router_type) for _ in range(cfg.num_layers)])
        self.ln_f = nn.LayerNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

    def forward(self, input_ids, step=0):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos_embed(pos)
        total_aux = 0.0
        all_indices = []
        for block in self.blocks:
            x, aux, indices = block(x, step)
            total_aux += aux
            all_indices.append(indices)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, total_aux, all_indices

# -------------------- METRICS (FIXED DEVICE) --------------------
def expert_utilization_entropy(all_indices, num_experts):
    if not all_indices:
        return 0.0
    device = all_indices[0].device
    counts = torch.zeros(num_experts, device=device)
    for indices in all_indices:
        for e in range(num_experts):
            counts[e] += (indices == e).float().sum()
    probs = counts / counts.sum()
    probs = probs.clamp(min=1e-8)
    entropy = -(probs * probs.log()).sum()
    max_entropy = torch.tensor(num_experts, device=device).float().log()
    return (entropy / max_entropy).item()

# -------------------- DATA LOADER --------------------
def get_dataloader(cfg, tokenizer, split="train"):
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split=split)
    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=cfg.max_seq_len,
                         padding="max_length", return_tensors="pt")
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
    tokenized.set_format("torch")
    return DataLoader(tokenized, batch_size=cfg.batch_size, shuffle=True, drop_last=True)

# -------------------- TRAINING --------------------
def train(router_type, cfg):
    torch.manual_seed(cfg.seed)
    print(f"\n{'='*50}")
    print(f"Training {router_type.upper()} router")
    print(f"{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = MoETransformer(cfg, router_type).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.total_steps)

    train_loader = get_dataloader(cfg, tokenizer, "train")
    val_loader   = get_dataloader(cfg, tokenizer, "validation")

    logs = []
    step = 0
    data_iter = iter(train_loader)

    while step < cfg.total_steps:
        model.train()
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(cfg.device)
        targets = input_ids[:, 1:].contiguous()
        inputs  = input_ids[:, :-1].contiguous()

        logits, aux_loss, all_indices = model(inputs, step=step)
        lm_loss = F.cross_entropy(logits.view(-1, cfg.vocab_size),
                                  targets.view(-1),
                                  ignore_index=tokenizer.pad_token_id)
        loss = lm_loss + 0.01 * aux_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()
        step += 1

        if step % cfg.log_every == 0 or step == cfg.total_steps:
            model.eval()
            val_losses = []
            val_entropies = []
            with torch.no_grad():
                for i, vbatch in enumerate(val_loader):
                    if i >= 10: break
                    vids = vbatch["input_ids"].to(cfg.device)
                    vi, vt = vids[:, :-1], vids[:, 1:]
                    vlogits, _, vindices = model(vi, step=step)
                    vloss = F.cross_entropy(vlogits.view(-1, cfg.vocab_size),
                                            vt.reshape(-1),
                                            ignore_index=tokenizer.pad_token_id)
                    val_losses.append(vloss.item())
                    val_entropies.append(expert_utilization_entropy(vindices, cfg.num_experts))
            ppl = np.exp(np.mean(val_losses))
            ent = np.mean(val_entropies)
            phase = "warmup" if step < cfg.warmup_steps else "bbr"
            logs.append({"step": step, "router": router_type, "perplexity": round(ppl,2),
                         "entropy": round(ent,4), "phase": phase})
            print(f"Step {step:5d} | PPL: {ppl:.2f} | Entropy: {ent:.4f} | {phase}")

    return logs

# -------------------- RUN --------------------
print("Starting Standard Router (this will take ~20 minutes)...")
logs_std = train("standard", cfg)
print("\nStarting BBR Router (another ~20 minutes)...")
logs_bbr = train("bbr", cfg)

results = {"standard": logs_std, "bbr": logs_bbr}
with open("/content/bbr_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*50)
print("Experiment Complete. Final comparison:")
print("Router    | Final Perplexity | Final Entropy")
print("-"*45)
for r in ["standard", "bbr"]:
    last = results[r][-1]
    print(f"{r:<9} | {last['perplexity']:>16.2f} | {last['entropy']:>12.4f}")

Starting Standard Router (this will take ~20 minutes)...

Training STANDARD router
Step   500 | PPL: 1352.57 | Entropy: 0.9957 | warmup
Step  1000 | PPL: 1395.59 | Entropy: 0.9971 | warmup
Step  1500 | PPL: 824.42 | Entropy: 0.9963 | warmup
Step  2000 | PPL: 371.66 | Entropy: 0.9981 | bbr
Step  2500 | PPL: 590.12 | Entropy: 0.9981 | bbr
Step  3000 | PPL: 492.80 | Entropy: 0.9980 | bbr
Step  3500 | PPL: 492.13 | Entropy: 0.9970 | bbr
Step  4000 | PPL: 405.49 | Entropy: 0.9960 | bbr
Step  4500 | PPL: nan | Entropy: 0.9974 | bbr
Step  5000 | PPL: 512.61 | Entropy: 0.9987 | bbr
Step  5500 | PPL: 258.49 | Entropy: 0.9971 | bbr
Step  6000 | PPL: 366.99 | Entropy: 0.9989 | bbr
Step  6500 | PPL: 842.73 | Entropy: 0.9988 | bbr
Step  7000 | PPL: 684.05 | Entropy: 0.9990 | bbr
Step  7500 | PPL: 312.17 | Entropy: 0.9990 | bbr
Step  8000 | PPL: 431.75 | Entropy: 0.9990 | bbr
Step  8500 | PPL: 456.10 | Entropy: 0.9993 | bbr
Step  9000 | PPL: 370.05 | Entropy: 0.9990 | bbr
Step  9500 | PPL: 431.34 | 

In [ ]:
from google.colab import files
files.download("/content/bbr_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**experiment - 2**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import json

# -------------------- CONFIG --------------------
class Config:
    hidden_size = 256
    num_layers = 2
    num_heads = 4
    num_experts = 16
    expert_hidden = 512
    max_seq_len = 64
    vocab_size = 50257

    d_embed = 16                     # = num_experts
    ema_alpha = 0.9
    sinkhorn_iters = 3
    sinkhorn_temp = 0.1
    capacity_slack = 0.2
    top_k_soft = 4

    batch_size = 8
    total_steps = 8000
    lr = 3e-4
    grad_clip = 1.0
    collapse_weight = 0.001

    device = "cuda"
    log_every = 500
    seed = 42

cfg = Config()

# -------------------- SINKHORN (soft, returns P) --------------------
def sinkhorn_soft(S, n_iters, capacity, temp):
    T, E = S.shape
    P = torch.exp(S / temp)
    for _ in range(n_iters):
        P = P / (P.sum(dim=1, keepdim=True) + 1e-8)
        col_sum = P.sum(dim=0)
        scale = torch.minimum(capacity / (col_sum + 1e-8), torch.ones_like(col_sum))
        P = P * scale.unsqueeze(0)
    return P

# -------------------- STANDARD ROUTER --------------------
class StandardRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.gate = nn.Linear(cfg.hidden_size, cfg.num_experts, bias=False)

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        indices = probs.argmax(dim=-1)
        T = x.shape[0]
        E = self.cfg.num_experts
        f = torch.zeros(E, device=x.device)
        for e in range(E):
            f[e] = (indices == e).float().mean()
        p = probs.mean(dim=0)
        aux_loss = E * (f * p).sum()
        return indices, probs, aux_loss

# -------------------- BBR ROUTER --------------------
class BBRRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_proj = nn.Linear(cfg.hidden_size, cfg.d_embed, bias=False)
        self.register_buffer('prototypes', F.normalize(torch.randn(cfg.num_experts, cfg.d_embed), dim=-1))

    def get_capacity(self, T):
        base = T / self.cfg.num_experts * (1 + self.cfg.capacity_slack)
        return torch.full((self.cfg.num_experts,), base, device=self.prototypes.device)

    def forward(self, x, return_soft_weights=False):
        T, H = x.shape
        tok_emb = F.normalize(self.token_proj(x), dim=-1)
        proto_emb = F.normalize(self.prototypes, dim=-1)
        S = tok_emb @ proto_emb.T
        capacity = self.get_capacity(T)
        P = sinkhorn_soft(S, self.cfg.sinkhorn_iters, capacity, self.cfg.sinkhorn_temp)

        # Update prototypes (EMA) using hard assignments (detached)
        with torch.no_grad():
            indices_hard = P.argmax(dim=-1)
            alpha = self.cfg.ema_alpha
            for e in range(self.cfg.num_experts):
                mask = (indices_hard == e)
                if mask.any():
                    new_proto = tok_emb[mask].mean(dim=0)
                    new_proto = F.normalize(new_proto, dim=-1)
                    self.prototypes[e] = alpha * self.prototypes[e] + (1 - alpha) * new_proto
                    self.prototypes[e] = F.normalize(self.prototypes[e], dim=-1)

        # Anti-collapse loss
        proto_n = F.normalize(self.prototypes, dim=-1)
        sim = proto_n @ proto_n.T
        mask_off = 1 - torch.eye(self.cfg.num_experts, device=x.device)
        collapse_loss = (sim * mask_off).sum() / (self.cfg.num_experts * (self.cfg.num_experts - 1))

        # Top‑k soft weights
        top_weights, top_indices = P.topk(self.cfg.top_k_soft, dim=-1)
        top_weights = top_weights / (top_weights.sum(dim=-1, keepdim=True) + 1e-8)

        if return_soft_weights:
            return top_indices, top_weights, collapse_loss
        else:
            return top_indices, top_weights, collapse_loss

# -------------------- MoE LAYER --------------------
class Expert(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.hidden_size, cfg.expert_hidden),
            nn.GELU(),
            nn.Linear(cfg.expert_hidden, cfg.hidden_size)
        )
    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.experts = nn.ModuleList([Expert(cfg) for _ in range(cfg.num_experts)])
        self.router_type = router_type
        if router_type == "standard":
            self.router = StandardRouter(cfg)
        else:
            self.router = BBRRouter(cfg)

    def forward(self, x):
        B, L, H = x.shape
        x_flat = x.view(B*L, H)
        if self.router_type == "standard":
            indices, probs, aux_loss = self.router(x_flat)
            out = torch.zeros_like(x_flat)
            for e in range(self.cfg.num_experts):
                mask = (indices == e)
                if mask.any():
                    out[mask] = self.experts[e](x_flat[mask])
            return out.view(B, L, H), aux_loss, indices
        else:  # BBR
            top_indices, top_weights, collapse_loss = self.router(x_flat, return_soft_weights=True)
            out = torch.zeros_like(x_flat)
            for k in range(self.cfg.top_k_soft):
                e_idx = top_indices[:, k]
                w = top_weights[:, k:k+1]
                for e in range(self.cfg.num_experts):
                    mask = (e_idx == e)
                    if mask.any():
                        out[mask] += w[mask] * self.experts[e](x_flat[mask])
            # For entropy logging, use primary assignment (top‑1)
            hard_indices = top_indices[:, 0]
            return out.view(B, L, H), collapse_loss, hard_indices

# -------------------- TRANSFORMER BLOCK --------------------
class TransformerBlock(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, batch_first=True)
        self.moe = MoELayer(cfg, router_type)
        self.ln1 = nn.LayerNorm(cfg.hidden_size)
        self.ln2 = nn.LayerNorm(cfg.hidden_size)

    def forward(self, x):
        L = x.shape[1]
        causal_mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = self.ln1(x + attn_out)
        moe_out, aux_loss, indices = self.moe(x)
        x = self.ln2(x + moe_out)
        return x, aux_loss, indices

class MoETransformer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_embed = nn.Embedding(cfg.max_seq_len, cfg.hidden_size)
        self.blocks = nn.ModuleList([TransformerBlock(cfg, router_type) for _ in range(cfg.num_layers)])
        self.ln_f = nn.LayerNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

    def forward(self, input_ids):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos_embed(pos)
        total_aux = 0.0
        all_indices = []
        for block in self.blocks:
            x, aux, indices = block(x)
            if aux is not None:
                total_aux += aux
            if indices is not None:
                all_indices.append(indices)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, total_aux, all_indices

# -------------------- METRICS --------------------
def expert_utilization_entropy(all_indices, num_experts):
    if not all_indices:
        return 0.0
    device = all_indices[0].device
    counts = torch.zeros(num_experts, device=device)
    for indices in all_indices:
        for e in range(num_experts):
            counts[e] += (indices == e).float().sum()
    probs = counts / counts.sum()
    probs = probs.clamp(min=1e-8)
    entropy = -(probs * probs.log()).sum()
    max_entropy = torch.tensor(num_experts, device=device).float().log()
    return (entropy / max_entropy).item()

def expert_load_stats(all_indices, num_experts):
    if not all_indices:
        return 0, 0, 0, 0
    device = all_indices[0].device
    counts = torch.zeros(num_experts, device=device)
    for indices in all_indices:
        for e in range(num_experts):
            counts[e] += (indices == e).float().sum()
    dead_experts = (counts == 0).sum().item()
    loads = counts.tolist()
    return max(loads), min(loads), np.std(loads), dead_experts

# -------------------- DATA --------------------
def get_dataloader(cfg, tokenizer, split="train"):
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split=split)
    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=cfg.max_seq_len,
                         padding="max_length", return_tensors="pt")
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
    tokenized.set_format("torch")
    return DataLoader(tokenized, batch_size=cfg.batch_size, shuffle=True, drop_last=True)

# -------------------- TRAINING --------------------
def train(router_type, cfg):
    torch.manual_seed(cfg.seed)
    print(f"\n{'='*50}")
    print(f"Training {router_type.upper()} router")
    print(f"{'='*50}")

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = MoETransformer(cfg, router_type).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.total_steps)

    train_loader = get_dataloader(cfg, tokenizer, "train")
    val_loader = get_dataloader(cfg, tokenizer, "validation")

    logs = []
    step = 0
    data_iter = iter(train_loader)

    while step < cfg.total_steps:
        model.train()
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(cfg.device)
        targets = input_ids[:, 1:].contiguous()
        inputs = input_ids[:, :-1].contiguous()

        logits, aux_loss, indices = model(inputs)
        lm_loss = F.cross_entropy(logits.view(-1, cfg.vocab_size),
                                  targets.view(-1),
                                  ignore_index=tokenizer.pad_token_id)
        if router_type == "standard":
            loss = lm_loss + 0.01 * aux_loss
        else:
            loss = lm_loss + cfg.collapse_weight * aux_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()
        step += 1

        if step % cfg.log_every == 0 or step == cfg.total_steps:
            model.eval()
            val_losses = []
            val_entropies = []
            all_load_stats = []
            with torch.no_grad():
                for i, vbatch in enumerate(val_loader):
                    if i >= 10: break
                    vids = vbatch["input_ids"].to(cfg.device)
                    vi, vt = vids[:, :-1], vids[:, 1:]
                    vlogits, _, vindices = model(vi)
                    vloss = F.cross_entropy(vlogits.view(-1, cfg.vocab_size),
                                            vt.reshape(-1),
                                            ignore_index=tokenizer.pad_token_id)
                    val_losses.append(vloss.item())
                    val_entropies.append(expert_utilization_entropy(vindices, cfg.num_experts))
                    all_load_stats.append(expert_load_stats(vindices, cfg.num_experts))
            ppl = np.exp(np.mean(val_losses))
            ent = np.mean(val_entropies)
            max_load, min_load, std_load, dead = map(np.mean, zip(*all_load_stats))
            logs.append({
                "step": step, "router": router_type,
                "perplexity": round(ppl,2), "entropy": round(ent,4),
                "max_load": round(max_load,2), "min_load": round(min_load,2),
                "std_load": round(std_load,2), "dead_experts": round(dead,2)
            })
            print(f"Step {step:5d} | PPL: {ppl:.2f} | Entropy: {ent:.4f} | Max/Min Load: {max_load:.0f}/{min_load:.0f} | Dead: {dead:.0f}")
    return logs

# -------------------- RUN --------------------
print("Training STANDARD router...")
logs_std = train("standard", cfg)
print("\nTraining BBR router...")
logs_bbr = train("bbr", cfg)

results = {"standard": logs_std, "bbr": logs_bbr}
with open("/content/bbr_final_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*50)
print("FINAL COMPARISON (16 experts, soft combination, top-4):")
print("Router    | Final PPL | Entropy | Max/Min Load | Dead Experts")
print("-"*65)
for r in ["standard", "bbr"]:
    last = results[r][-1]
    print(f"{r:<9} | {last['perplexity']:>8.2f} | {last['entropy']:.4f}  | {last['max_load']:.0f}/{last['min_load']:.0f}      | {last['dead_experts']:.0f}")

Training STANDARD router...

Training STANDARD router


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Step   500 | PPL: 1246.80 | Entropy: 0.9950 | Max/Min Load: 86/47 | Dead: 0
Step  1000 | PPL: 669.55 | Entropy: 0.9963 | Max/Min Load: 82/48 | Dead: 0
Step  1500 | PPL: 1130.32 | Entropy: 0.9957 | Max/Min Load: 82/47 | Dead: 0
Step  2000 | PPL: 817.18 | Entropy: 0.9967 | Max/Min Load: 80/47 | Dead: 0
Step  2500 | PPL: 534.10 | Entropy: 0.9958 | Max/Min Load: 82/48 | Dead: 0
Step  3000 | PPL: 672.52 | Entropy: 0.9969 | Max/Min Load: 78/47 | Dead: 0
Step  3500 | PPL: 609.79 | Entropy: 0.9974 | Max/Min Load: 75/49 | Dead: 0
Step  4000 | PPL: 604.96 | Entropy: 0.9954 | Max/Min Load: 85/47 | Dead: 0
Step  4500 | PPL: 660.68 | Entropy: 0.9971 | Max/Min Load: 76/48 | Dead: 0
Step  5000 | PPL: 662.18 | Entropy: 0.9976 | Max/Min Load: 76/50 | Dead: 0
Step  5500 | PPL: 543.82 | Entropy: 0.9985 | Max/Min Load: 74/53 | Dead: 0
Step  6000 | PPL: 419.39 | Entropy: 0.9984 | Max/Min Load: 75/54 | Dead: 0
Step  6500 | PPL: 536.66 | Entropy: 0.9982 | Max/Min Load: 75/52 | Dead: 0
Step  7000 | PPL: 446.1

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Step   500 | PPL: 1052.72 | Entropy: 0.9642 | Max/Min Load: 109/18 | Dead: 0
Step  1000 | PPL: 959.79 | Entropy: 0.9687 | Max/Min Load: 110/21 | Dead: 0
Step  1500 | PPL: 694.74 | Entropy: 0.9774 | Max/Min Load: 108/31 | Dead: 0
Step  2000 | PPL: 683.03 | Entropy: 0.9721 | Max/Min Load: 112/25 | Dead: 0
Step  2500 | PPL: 630.49 | Entropy: 0.9680 | Max/Min Load: 124/22 | Dead: 0
Step  3000 | PPL: 551.66 | Entropy: 0.9757 | Max/Min Load: 117/26 | Dead: 0
Step  3500 | PPL: 495.56 | Entropy: 0.9674 | Max/Min Load: 114/22 | Dead: 0
Step  4000 | PPL: 503.24 | Entropy: 0.9710 | Max/Min Load: 113/23 | Dead: 0
Step  4500 | PPL: 351.56 | Entropy: 0.9694 | Max/Min Load: 116/23 | Dead: 0
Step  5000 | PPL: 434.93 | Entropy: 0.9609 | Max/Min Load: 126/20 | Dead: 0
Step  5500 | PPL: 377.67 | Entropy: 0.9743 | Max/Min Load: 110/28 | Dead: 0
Step  6000 | PPL: 489.78 | Entropy: 0.9753 | Max/Min Load: 107/23 | Dead: 0
Step  6500 | PPL: 432.48 | Entropy: 0.9726 | Max/Min Load: 108/24 | Dead: 0
Step  7000 

Experiment-3


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import json

# -------------------- CONFIG --------------------
class Config:
    hidden_size = 256
    num_layers = 2
    num_heads = 4
    num_experts = 16
    expert_hidden = 512
    max_seq_len = 64
    vocab_size = 50257

    d_embed = 16
    ema_alpha = 0.9
    sinkhorn_iters = 3
    sinkhorn_temp = 0.5        # FIX 1: was 0.1 — softer assignments, capacity actually enforces
    capacity_slack = 0.2
    top_k_soft = 4             # both routers use top-4 (fair comparison)

    batch_size = 8
    total_steps = 8000
    lr = 3e-4
    grad_clip = 1.0
    collapse_weight = 0.001

    device = "cuda"
    log_every = 500
    seed = 42

cfg = Config()

# -------------------- SINKHORN --------------------
def sinkhorn_soft(S, n_iters, capacity, temp):
    """
    Capacity-constrained Sinkhorn-Knopp.
    S: [T, E] similarity matrix
    capacity: [E] max tokens per expert
    Returns P: [T, E] soft assignment matrix
    """
    T, E = S.shape
    P = torch.exp(S / temp)
    for _ in range(n_iters):
        # Row normalize: each token sums to 1
        P = P / (P.sum(dim=1, keepdim=True) + 1e-8)
        # Column cap: enforce expert capacity
        col_sum = P.sum(dim=0)
        scale = torch.minimum(capacity / (col_sum + 1e-8), torch.ones_like(col_sum))
        P = P * scale.unsqueeze(0)
    return P  # [T, E]

# -------------------- STANDARD ROUTER (FIX 2: top-4 soft, fair baseline) --------------------
class StandardRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.gate = nn.Linear(cfg.hidden_size, cfg.num_experts, bias=False)

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)   # [T, E]

        # Top-4 soft combination (same compute as BBR — fair comparison)
        top_weights, top_indices = probs.topk(self.cfg.top_k_soft, dim=-1)  # [T, 4]
        top_weights = top_weights / top_weights.sum(dim=-1, keepdim=True)   # renormalize

        # Aux loss still uses full distribution (standard Switch style)
        E = self.cfg.num_experts
        indices_hard = probs.argmax(dim=-1)
        f = torch.zeros(E, device=x.device)
        for e in range(E):
            f[e] = (indices_hard == e).float().mean()
        p = probs.mean(dim=0)
        aux_loss = E * (f * p).sum()

        return top_indices, top_weights, aux_loss

# -------------------- BBR ROUTER --------------------
class BBRRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_proj = nn.Linear(cfg.hidden_size, cfg.d_embed, bias=False)
        self.register_buffer(
            'prototypes',
            F.normalize(torch.randn(cfg.num_experts, cfg.d_embed), dim=-1)
        )

    def get_capacity(self, T):
        base = T / self.cfg.num_experts * (1 + self.cfg.capacity_slack)
        return torch.full((self.cfg.num_experts,), base, device=self.prototypes.device)

    def forward(self, x):
        T, H = x.shape
        E = self.cfg.num_experts

        # Project + normalize
        tok_emb = F.normalize(self.token_proj(x), dim=-1)   # [T, d]
        proto_emb = F.normalize(self.prototypes, dim=-1)    # [E, d]

        # Similarity + Sinkhorn
        S = tok_emb @ proto_emb.T                           # [T, E]
        capacity = self.get_capacity(T)
        P = sinkhorn_soft(S, self.cfg.sinkhorn_iters, capacity, self.cfg.sinkhorn_temp)

        # EMA prototype update (detached, no grad)
        with torch.no_grad():
            indices_hard = P.argmax(dim=-1)                 # [T]
            alpha = self.cfg.ema_alpha
            for e in range(E):
                mask = (indices_hard == e)
                if mask.any():
                    new_proto = tok_emb[mask].mean(dim=0)
                    new_proto = F.normalize(new_proto, dim=-1)
                    self.prototypes[e] = alpha * self.prototypes[e] + (1 - alpha) * new_proto
                    self.prototypes[e] = F.normalize(self.prototypes[e], dim=-1)

        # Anti-collapse loss: push prototypes apart
        proto_n = F.normalize(self.prototypes, dim=-1)
        sim = proto_n @ proto_n.T                           # [E, E]
        mask_off = 1 - torch.eye(E, device=x.device)
        collapse_loss = (sim * mask_off).sum() / (E * (E - 1))

        # Top-k soft weights
        top_weights, top_indices = P.topk(self.cfg.top_k_soft, dim=-1)  # [T, 4]
        top_weights = top_weights / (top_weights.sum(dim=-1, keepdim=True) + 1e-8)

        return top_indices, top_weights, collapse_loss

# -------------------- EXPERT --------------------
class Expert(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.hidden_size, cfg.expert_hidden),
            nn.GELU(),
            nn.Linear(cfg.expert_hidden, cfg.hidden_size)
        )
    def forward(self, x):
        return self.net(x)

# -------------------- MoE LAYER (FIX 3: both routers use identical soft forward) --------------------
class MoELayer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.experts = nn.ModuleList([Expert(cfg) for _ in range(cfg.num_experts)])
        self.router_type = router_type
        if router_type == "standard":
            self.router = StandardRouter(cfg)
        else:
            self.router = BBRRouter(cfg)

    def forward(self, x):
        B, L, H = x.shape
        x_flat = x.view(B * L, H)   # [T, H]

        # Both routers return (top_indices [T,4], top_weights [T,4], aux_loss)
        top_indices, top_weights, aux_loss = self.router(x_flat)

        # Identical soft combination for both routers
        out = torch.zeros_like(x_flat)
        for k in range(self.cfg.top_k_soft):
            e_idx = top_indices[:, k]       # [T]
            w = top_weights[:, k:k+1]       # [T, 1]
            for e in range(self.cfg.num_experts):
                mask = (e_idx == e)
                if mask.any():
                    out[mask] += w[mask] * self.experts[e](x_flat[mask])

        # Primary assignment for entropy logging
        hard_indices = top_indices[:, 0]    # [T]
        return out.view(B, L, H), aux_loss, hard_indices

# -------------------- TRANSFORMER BLOCK --------------------
class TransformerBlock(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads, batch_first=True)
        self.moe = MoELayer(cfg, router_type)
        self.ln1 = nn.LayerNorm(cfg.hidden_size)
        self.ln2 = nn.LayerNorm(cfg.hidden_size)

    def forward(self, x):
        L = x.shape[1]
        causal_mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = self.ln1(x + attn_out)
        moe_out, aux_loss, indices = self.moe(x)
        x = self.ln2(x + moe_out)
        return x, aux_loss, indices

# -------------------- TRANSFORMER --------------------
class MoETransformer(nn.Module):
    def __init__(self, cfg, router_type):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_embed = nn.Embedding(cfg.max_seq_len, cfg.hidden_size)
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg, router_type) for _ in range(cfg.num_layers)
        ])
        self.ln_f = nn.LayerNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

    def forward(self, input_ids):
        B, L = input_ids.shape
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos_embed(pos)
        total_aux = 0.0
        all_indices = []
        for block in self.blocks:
            x, aux, indices = block(x)
            if aux is not None:
                total_aux += aux
            if indices is not None:
                all_indices.append(indices)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, total_aux, all_indices

# -------------------- METRICS --------------------
def expert_utilization_entropy(all_indices, num_experts):
    if not all_indices:
        return 0.0
    device = all_indices[0].device
    counts = torch.zeros(num_experts, device=device)
    for indices in all_indices:
        for e in range(num_experts):
            counts[e] += (indices == e).float().sum()
    probs = counts / counts.sum()
    probs = probs.clamp(min=1e-8)
    entropy = -(probs * probs.log()).sum()
    max_entropy = torch.tensor(num_experts, device=device).float().log()
    return (entropy / max_entropy).item()

def expert_load_stats(all_indices, num_experts):
    if not all_indices:
        return 0, 0, 0, 0
    device = all_indices[0].device
    counts = torch.zeros(num_experts, device=device)
    for indices in all_indices:
        for e in range(num_experts):
            counts[e] += (indices == e).float().sum()
    dead_experts = (counts == 0).sum().item()
    loads = counts.tolist()
    return max(loads), min(loads), float(np.std(loads)), dead_experts

# -------------------- DATA --------------------
def get_dataloader(cfg, tokenizer, split="train"):
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split=split)
    def tokenize(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=cfg.max_seq_len,
            padding="max_length",
            return_tensors="pt"
        )
    tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
    tokenized.set_format("torch")
    return DataLoader(tokenized, batch_size=cfg.batch_size, shuffle=True, drop_last=True)

# -------------------- TRAINING --------------------
def train(router_type, cfg):
    torch.manual_seed(cfg.seed)
    print(f"\n{'='*55}")
    print(f"  Training {router_type.upper()} router  |  top-4 soft  |  16 experts")
    print(f"{'='*55}")

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = MoETransformer(cfg, router_type).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.total_steps)

    train_loader = get_dataloader(cfg, tokenizer, "train")
    val_loader   = get_dataloader(cfg, tokenizer, "validation")

    logs = []
    step = 0
    data_iter = iter(train_loader)

    while step < cfg.total_steps:
        model.train()
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(cfg.device)
        targets = input_ids[:, 1:].contiguous()
        inputs  = input_ids[:, :-1].contiguous()

        logits, aux_loss, _ = model(inputs)
        lm_loss = F.cross_entropy(
            logits.view(-1, cfg.vocab_size),
            targets.view(-1),
            ignore_index=tokenizer.pad_token_id
        )

        if router_type == "standard":
            loss = lm_loss + 0.01 * aux_loss
        else:
            loss = lm_loss + cfg.collapse_weight * aux_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()
        step += 1

        if step % cfg.log_every == 0 or step == cfg.total_steps:
            model.eval()
            val_losses, val_entropies, load_stats_list = [], [], []
            with torch.no_grad():
                for i, vbatch in enumerate(val_loader):
                    if i >= 10:
                        break
                    vids = vbatch["input_ids"].to(cfg.device)
                    vi, vt = vids[:, :-1], vids[:, 1:]
                    vlogits, _, vindices = model(vi)
                    vloss = F.cross_entropy(
                        vlogits.view(-1, cfg.vocab_size),
                        vt.reshape(-1),
                        ignore_index=tokenizer.pad_token_id
                    )
                    val_losses.append(vloss.item())
                    val_entropies.append(
                        expert_utilization_entropy(vindices, cfg.num_experts)
                    )
                    load_stats_list.append(
                        expert_load_stats(vindices, cfg.num_experts)
                    )

            ppl = np.exp(np.mean(val_losses))
            ent = np.mean(val_entropies)
            max_load, min_load, std_load, dead = map(np.mean, zip(*load_stats_list))

            logs.append({
                "step": step,
                "router": router_type,
                "perplexity": round(float(ppl), 2),
                "entropy": round(float(ent), 4),
                "max_load": round(float(max_load), 2),
                "min_load": round(float(min_load), 2),
                "std_load": round(float(std_load), 2),
                "dead_experts": round(float(dead), 2)
            })
            print(
                f"Step {step:5d} | PPL: {ppl:7.2f} | Entropy: {ent:.4f} "
                f"| Max/Min: {max_load:.0f}/{min_load:.0f} | Dead: {dead:.0f}"
            )

    return logs

# -------------------- RUN --------------------
print("=" * 55)
print("  FINAL BBR EXPERIMENT — Fair Comparison (top-4 both)")
print("  Fix 1: Sinkhorn temp 0.1 → 0.5")
print("  Fix 2: Standard router now uses top-4 soft (equal compute)")
print("  Fix 3: Both routers use identical MoE forward pass")
print("=" * 55)

print("\nTraining STANDARD router...")
logs_std = train("standard", cfg)

print("\nTraining BBR router...")
logs_bbr = train("bbr", cfg)

results = {"standard": logs_std, "bbr": logs_bbr}
with open("/content/bbr_final_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 65)
print("FINAL COMPARISON (16 experts, top-4 soft, FAIR baseline):")
print(f"{'Router':<10} | {'PPL':>8} | {'Entropy':>7} | {'Max/Min':>8} | {'Dead':>4}")
print("-" * 65)
for r in ["standard", "bbr"]:
    last = results[r][-1]
    print(
        f"{r:<10} | {last['perplexity']:>8.2f} | {last['entropy']:>7.4f} "
        f"| {last['max_load']:.0f}/{last['min_load']:.0f}     | {last['dead_experts']:>4.0f}"
    )

# Download results (Colab)
try:
    from google.colab import files
    files.download("/content/bbr_final_results.json")
except Exception:
    print("\nResults saved to /content/bbr_final_results.json")

  FINAL BBR EXPERIMENT — Fair Comparison (top-4 both)
  Fix 1: Sinkhorn temp 0.1 → 0.5
  Fix 2: Standard router now uses top-4 soft (equal compute)
  Fix 3: Both routers use identical MoE forward pass

Training STANDARD router...

  Training STANDARD router  |  top-4 soft  |  16 experts


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Step   500 | PPL: 1130.67 | Entropy: 0.9739 | Max/Min: 144/35 | Dead: 0
Step  1000 | PPL:  598.62 | Entropy: 0.9843 | Max/Min: 112/33 | Dead: 0
Step  1500 | PPL:  961.79 | Entropy: 0.9887 | Max/Min: 99/37 | Dead: 0
Step  2000 | PPL:  696.84 | Entropy: 0.9895 | Max/Min: 100/42 | Dead: 0
Step  2500 | PPL:  435.59 | Entropy: 0.9859 | Max/Min: 105/36 | Dead: 0
Step  3000 | PPL:  551.23 | Entropy: 0.9894 | Max/Min: 97/42 | Dead: 0
Step  3500 | PPL:  521.26 | Entropy: 0.9912 | Max/Min: 96/44 | Dead: 0
Step  4000 | PPL:  492.79 | Entropy: 0.9905 | Max/Min: 91/41 | Dead: 0
Step  4500 | PPL:  545.35 | Entropy: 0.9958 | Max/Min: 82/46 | Dead: 0
Step  5000 | PPL:  537.57 | Entropy: 0.9899 | Max/Min: 97/40 | Dead: 0
Step  5500 | PPL:  446.97 | Entropy: 0.9902 | Max/Min: 92/38 | Dead: 0
Step  6000 | PPL:  350.90 | Entropy: 0.9918 | Max/Min: 92/42 | Dead: 0
Step  6500 | PPL:  438.23 | Entropy: 0.9920 | Max/Min: 88/44 | Dead: 0
Step  7000 | PPL:  366.56 | Entropy: 0.9952 | Max/Min: 83/48 | Dead: 0
St

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Step   500 | PPL: 1058.21 | Entropy: 0.9686 | Max/Min: 116/27 | Dead: 0
Step  1000 | PPL:  961.88 | Entropy: 0.9707 | Max/Min: 120/28 | Dead: 0
Step  1500 | PPL:  699.30 | Entropy: 0.9756 | Max/Min: 104/26 | Dead: 0
Step  2000 | PPL:  678.61 | Entropy: 0.9692 | Max/Min: 123/25 | Dead: 0
Step  2500 | PPL:  628.59 | Entropy: 0.9758 | Max/Min: 113/29 | Dead: 0
Step  3000 | PPL:  546.64 | Entropy: 0.9760 | Max/Min: 106/33 | Dead: 0
Step  3500 | PPL:  497.79 | Entropy: 0.9747 | Max/Min: 99/30 | Dead: 0
Step  4000 | PPL:  506.79 | Entropy: 0.9726 | Max/Min: 103/24 | Dead: 0
Step  4500 | PPL:  360.42 | Entropy: 0.9720 | Max/Min: 105/29 | Dead: 0
Step  5000 | PPL:  440.24 | Entropy: 0.9677 | Max/Min: 109/29 | Dead: 0
Step  5500 | PPL:  378.32 | Entropy: 0.9810 | Max/Min: 102/32 | Dead: 0
Step  6000 | PPL:  499.74 | Entropy: 0.9786 | Max/Min: 107/30 | Dead: 0
Step  6500 | PPL:  442.14 | Entropy: 0.9762 | Max/Min: 104/30 | Dead: 0
Step  7000 | PPL:  319.73 | Entropy: 0.9512 | Max/Min: 125/26 | D

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>